# Actor-Critic Teorisi

Actor-Critic, policy gradient'in yüksek varyansını azaltmak için bir critic kullanır. Actor politikayı öğrenir; critic ise actor'a hangi aksiyonların beklenenden iyi olduğunu söyler.

Bu notebooktaki klasik Actor-Critic anlatımı on-policy kabul edilir: rollout verisi güncel actor'dan gelir, critic de aynı politikanın value fonksiyonunu tahmin eder. Off-policy actor-critic yöntemleri de vardır, fakat onlar importance sampling, replay correction veya ayrı critic tasarımları gerektirir.

## 1. İki Bileşen

- Actor: $\pi_\theta(a\mid s)$, aksiyon dağılımını üretir.
- Critic: $V_\phi(s)$ veya $Q_\phi(s,a)$, öğrenme sinyalini tahmin eder.

En yaygın form value critic kullanır:

$$V_\phi(s)\approx\mathbb{E}_{\pi}[G_t\mid S_t=s]$$

Actor şu sinyalle güncellenir:

$$\nabla_\theta J(\theta)\approx\nabla_\theta\log\pi_\theta(a_t\mid s_t)\hat{A}_t$$

## 2. TD Error Advantage Olarak Kullanılabilir

Tek adımlı TD residual:

$$\delta_t=r_{t+1}+\gamma(1-d_t)V_\phi(s_{t+1})-V_\phi(s_t)$$

Bu, advantage için düşük maliyetli bir tahmindir:

$$\hat{A}_t\approx\delta_t$$

Actor kaybı:

$$L_\pi=-\log\pi_\theta(a_t\mid s_t)\delta_t$$

Critic kaybı:

$$L_V=\frac{1}{2}\delta_t^2$$

TD yaklaşımı Monte Carlo'ya göre daha düşük varyanslıdır, fakat critic hatalıysa bias ekler.

### 2.1. TD Target, $\delta_t$ ve Advantage Tablosu

| Parça | Formül | Ne işe yarar? |
| --- | --- | --- |
| TD target | $r_{t+1}+\gamma(1-d_t)V_\phi(s_{t+1})$ | Critic'in $V(s_t)$ tahmini için bootstrap hedefi |
| Mevcut value | $V_\phi(s_t)$ | Critic'in şu anki durum değeri tahmini |
| TD residual | $\delta_t=TD\ target-V_\phi(s_t)$ | Critic hatası ve tek-adım advantage tahmini |
| Actor sinyali | $\hat{A}_t\approx\delta_t$ | Seçilen aksiyonun olasılığını artırma/azaltma yönü |
| Critic loss | $\frac{1}{2}\delta_t^2$ | Value tahminini TD target'a yaklaştırır |

Actor güncellemesinde $\delta_t$ genellikle `detach` edilir; actor critic'in hatasını öğrenme sinyali olarak kullanır, ama policy loss üzerinden critic parametrelerini doğrudan sürüklemez.

In [ ]:
import torch

def one_step_actor_critic_loss(log_probs, rewards, dones, values, next_values, gamma=0.99):
    td_target = rewards + gamma * (1.0 - dones) * next_values
    advantage = td_target - values
    policy_loss = -(log_probs * advantage.detach()).mean()
    value_loss = 0.5 * advantage.pow(2).mean()
    return {"td_target": td_target, "advantage": advantage, "policy_loss": policy_loss, "value_loss": value_loss}

log_probs = torch.tensor([-0.8, -1.5, -0.3])
rewards = torch.tensor([0.0, 1.0, -1.0])
dones = torch.tensor([0.0, 0.0, 1.0])
values = torch.tensor([0.2, 0.5, 0.1])
next_values = torch.tensor([0.5, 0.1, 0.0])
one_step_actor_critic_loss(log_probs, rewards, dones, values, next_values)

## 2.2. Minimal Actor-Critic Neural Network

Actor-Critic ağında ortak bir temsil iki başlığa ayrılır:

- Actor head: $\pi_	heta(a\mid s)$ için logits üretir.
- Critic head: $V_\phi(s)$ üretir.

Tek adımlı TD residual hem actor için advantage sinyali hem critic için hata sinyalidir.

In [ ]:
try:
    import torch
    import torch.nn as nn
    from torch.distributions import Categorical
except ModuleNotFoundError as exc:
    raise ModuleNotFoundError("Bu hücre için PyTorch gerekli. Jupyter'da orbit_wars_env kernelini seçin.") from exc

class MinimalActorCritic(nn.Module):
    """Ortak encoder + policy head + value head."""
    def __init__(self, obs_dim=12, hidden_dim=64, n_actions=5):
        super().__init__()
        self.encoder = nn.Sequential(
            nn.Linear(obs_dim, hidden_dim), nn.Tanh(),
            nn.Linear(hidden_dim, hidden_dim), nn.Tanh(),
        )
        self.actor = nn.Linear(hidden_dim, n_actions)
        self.critic = nn.Linear(hidden_dim, 1)

    def forward(self, obs):
        h = self.encoder(obs)
        logits = self.actor(h)
        value = self.critic(h).squeeze(-1)
        return logits, value

net = MinimalActorCritic()
obs = torch.randn(4, 12)
next_obs = torch.randn(4, 12)
rewards = torch.tensor([0.0, 1.0, -0.5, 0.2])
dones = torch.tensor([0.0, 0.0, 1.0, 0.0])
gamma = 0.99

logits, values = net(obs)
dist = Categorical(logits=logits)
actions = dist.sample()
log_probs = dist.log_prob(actions)
with torch.no_grad():
    _, next_values = net(next_obs)
    td_target = rewards + gamma * (1.0 - dones) * next_values

advantage = td_target - values
policy_loss = -(log_probs * advantage.detach()).mean()
value_loss = 0.5 * advantage.pow(2).mean()
entropy_bonus = dist.entropy().mean()
total_loss = policy_loss + 0.5 * value_loss - 0.01 * entropy_bonus

print("actions:", actions)
print("td_target:", td_target)
print("advantage:", advantage)
print("total_loss:", float(total_loss.detach()))

## 2.3. Actor-Critic Update Algoritması

Actor-Critic update'i aynı minibatch içinde üç sinyal kullanır:

- policy loss: actor'ı advantage yönünde günceller
- value loss: critic'i TD target'a yaklaştırır
- entropy bonus: politikayı erken deterministikleşmekten korur

In [ ]:
def actor_critic_train_step(net, optimizer, obs, actions, rewards, next_obs, dones, gamma=0.99, value_coef=0.5, entropy_coef=0.01):
    logits, values = net(obs)
    dist = Categorical(logits=logits)
    log_probs = dist.log_prob(actions)

    with torch.no_grad():
        _, next_values = net(next_obs)
        td_target = rewards + gamma * (1.0 - dones) * next_values

    advantages = td_target - values
    policy_loss = -(log_probs * advantages.detach()).mean()
    value_loss = 0.5 * advantages.pow(2).mean()
    entropy = dist.entropy().mean()
    loss = policy_loss + value_coef * value_loss - entropy_coef * entropy

    optimizer.zero_grad(set_to_none=True)
    loss.backward()
    torch.nn.utils.clip_grad_norm_(net.parameters(), max_norm=0.5)
    optimizer.step()

    return {
        "loss": float(loss.detach()),
        "policy_loss": float(policy_loss.detach()),
        "value_loss": float(value_loss.detach()),
        "entropy": float(entropy.detach()),
    }

net = MinimalActorCritic(obs_dim=12, n_actions=5)
optimizer = torch.optim.Adam(net.parameters(), lr=1e-3)
obs = torch.randn(8, 12)
next_obs = torch.randn(8, 12)
with torch.no_grad():
    actions = Categorical(logits=net(obs)[0]).sample()
rewards = torch.randn(8)
dones = torch.randint(0, 2, (8,)).float()
actor_critic_train_step(net, optimizer, obs, actions, rewards, next_obs, dones)

## 3. N-Step Actor-Critic

Tek adımlı TD çok düşük varyanslı ama yüksek bias'lı olabilir. N-step return ara bir çözümdür:

$$R_t^{(n)}=r_{t+1}+\gamma r_{t+2}+...+\gamma^{n-1}r_{t+n}+\gamma^nV(s_{t+n})$$

Advantage:

$$\hat{A}_t=R_t^{(n)}-V(s_t)$$

A2C/A3C gibi yöntemlerde n-step return yaygındır.

## 4. GAE ile Actor-Critic

Generalized Advantage Estimation, farklı uzunlukta TD residual'ları üstel ağırlıkla birleştirir:

$$\hat{A}_t=\sum_{l=0}^{\infty}(\gamma\lambda)^l\delta_{t+l}$$

$\lambda$ arttıkça Monte Carlo'ya yaklaşılır. $\lambda$ azaldıkça tek adımlı TD'ye yaklaşılır. PPO gibi modern actor-critic algoritmalarda GAE standart tercihtir.

## 5. Entropy Regularization

Actor-Critic'te politika erken deterministikleşebilir. Entropy bonus bunu yavaşlatır:

$$L=L_\pi+c_vL_V-c_eH(\pi)$$

Entropy katsayısı keşif-sömürü dengesini etkiler. Orbit Wars gibi self-play oyunlarda çok düşük entropy ajanı erken tek stratejiye kilitleyebilir; çok yüksek entropy ise tutarlı strateji öğrenmeyi zorlaştırabilir.

## 6. Orbit Wars Bağlantısı

`model.py::OrbitWarsPolicy` aynı encoder'dan hem actor başlıklarını hem value head'i üretir:

- Actor başlıkları: `src_logits`, `tgt_logits`, `ship_logits`
- Critic başlığı: `value`

Joint log-prob branch log-prob'larının toplamıdır:

$$\log\pi(a\mid s)=\log\pi_{src}+\log\pi_{tgt}+\log\pi_{ship}$$

`rl/ppo.py` içindeki PPO update'i aslında clipped actor-critic güncellemesidir: policy loss actor'ı, value loss critic'i, entropy bonus keşfi kontrol eder.

## 7. Actor-Critic'te Tipik Hatalar

| Belirti | Muhtemel neden |
| --- | --- |
| Value loss patlıyor | Reward ölçeği, learning rate veya bootstrap hatası |
| Entropy hızla sıfıra düşüyor | Keşif yetersiz, entropy coef düşük |
| Policy loss anlamsız dalgalı | Advantage normalize edilmiyor veya rollout çok küçük |
| Critic iyi, actor gelişmiyor | Advantage sinyali zayıf veya clipping/KL çok sert |
| Actor iyi başlıyor sonra çöküyor | Update adımı fazla agresif veya self-play dağılımı kayıyor |